# 16_landscape: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/16_landscape/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Reject benchmark comparisons that changed the workload.

Run: uv run python modules/16_landscape/examples/01_from_scratch.py
Concepts: Reject benchmark comparisons that changed the workload.
Expected: mismatched comparison inputs.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""


def differences(first, second):
    return [
        key
        for key in ("dataset", "queries", "k", "metric", "concurrency", "hardware", "recall_target")
        if first.get(key) != second.get(key)
    ]


def main() -> None:
    # KEY IDEA: Reject benchmark comparisons that changed the workload.
    a = {
        "dataset": "Marlowe 500",
        "queries": "test",
        "k": 10,
        "metric": "cosine",
        "concurrency": 1,
        "hardware": "same laptop",
        "recall_target": 0.95,
    }
    b = {**a, "concurrency": 16, "recall_target": 0.8}
    print("incompatible fields", differences(a, b))
    assert differences(a, b) == ["concurrency", "recall_target"]


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/16_landscape/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("fastapi") is None or importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra apps --extra libs")
else:
    """Serve FAISS behind a local request and restore it in a new application.

    Run: uv run --extra apps --extra libs python modules/16_landscape/examples/02_production.py
    Concepts: Serve FAISS behind a local request and restore it in a new application.
    Expected: FAISS API lifecycle.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: apps,libs
    import tempfile
    from pathlib import Path

    from fastapi.testclient import TestClient

    from common.apps.faiss_service import create_app
    from common.teaching import catalog

    def main() -> None:
        # KEY IDEA: Serve FAISS behind a local request and restore it in a new application.
        products, matrix, q, meta = catalog(80)
        payload = [
            {"id": p.id, "vector": v.tolist(), "metadata": {"region": p.region}, "text": p.title}
            for p, v in zip(products.itertuples(), matrix, strict=True)
        ]
        request = {
            "vector": q.tolist(),
            "space": meta["space"],
            "k": 5,
            "filter": {"op": "eq", "field": "region", "value": "north"},
        }
        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp)
            with TestClient(create_app(path, 384, meta["space"])) as api:
                api.put("/items", json=payload).raise_for_status()
                first = api.post("/search", json=request).raise_for_status().json()
                api.post("/snapshot").raise_for_status()
            with TestClient(create_app(path, 384, meta["space"])) as api:
                restored = api.post("/search", json=request).raise_for_status().json()
                assert first == restored and all(r["metadata"]["region"] == "north" for r in first)
                print("HTTP contract restored", first)

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/16_landscape/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Measure a controlled database comparison on the same catalog.

    Run: uv run --extra embedded python modules/16_landscape/examples/03_measure.py
    Concepts: Measure a controlled database comparison on the same catalog.
    Expected: two matched embedded databases.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    from common.tier3 import measure

    def main() -> None:
        # KEY IDEA: Measure a controlled database comparison on the same catalog.
        for backend in ("duckdb", "sqlite-vec"):
            result = measure(backend)
            print(backend, result["summary"]["qps"], result["hardware_id"])

    if __name__ == "__main__":
        main()

## 04_elasticsearch

Source: `modules/16_landscape/examples/04_elasticsearch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("elasticsearch") is None or "elasticsearch" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra landscape; start elasticsearch and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Elasticsearch HNSW and a keyword query.

    Run: uv run --extra landscape python modules/16_landscape/examples/04_elasticsearch.py
    Concepts: Elasticsearch HNSW and a keyword query.
    Expected: Elasticsearch dense and text hits.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the elasticsearch profile first.
    """

    # course-extra: landscape
    # course-profile: elasticsearch
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Elasticsearch HNSW and a keyword query.
        with session("elasticsearch", 80) as (s, _, q):
            rows = s.db.search(
                index=s.name, query={"match": {"body": "mug warmer"}}, size=3, source=False
            )["hits"]["hits"]
            print("lexical IDs", [r["_id"] for r in rows])
            hits = s.search(q, 5, {"op": "eq", "field": "region", "value": "north"})
            assert hits and all(h.metadata["region"] == "north" for h in hits)
            print("vector hits", [h.id for h in hits])

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "elasticsearch")

## 05_redis

Source: `modules/16_landscape/examples/05_redis.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("elasticsearch") is None or "redis" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra landscape; start redis and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Redis Search native vector and text indexes.

    Run: uv run --extra landscape python modules/16_landscape/examples/05_redis.py
    Concepts: Redis Search native vector and text indexes.
    Expected: Redis tag and KNN.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the redis profile first.
    """

    # course-extra: landscape
    # course-profile: redis
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Redis Search native vector and text indexes.
        with session("redis", 80) as (s, _, q):
            rows = s.db.execute_command(
                "FT.SEARCH", s.name, "@body:(mug warmer)", "NOCONTENT", "LIMIT", 0, 3
            )
            print("lexical matches", rows)
            hits = s.search(q, 5, {"op": "eq", "field": "region", "value": "north"})
            assert hits and all(h.metadata["region"] == "north" for h in hits)
            print("vector hits", [h.id for h in hits])

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "redis")

## 06_mongodb

Source: `modules/16_landscape/examples/06_mongodb.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("elasticsearch") is None or "mongodb" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra landscape; start mongodb and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Atlas Local search-index readiness and exact vector retrieval.

    Run: uv run --extra landscape python modules/16_landscape/examples/06_mongodb.py
    Concepts: Atlas Local search-index readiness and exact vector retrieval.
    Expected: Atlas Local queryable index.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the mongodb profile first.
    """

    # course-extra: landscape
    # course-profile: mongodb
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Atlas Local search-index readiness and exact vector retrieval.
        with session("mongodb", 80) as (s, _, q):
            indexes = list(s.table.list_search_indexes("vectors"))
            assert indexes[0]["queryable"]
            print("native search index", indexes[0]["status"])
            hits = s.search(q, 5, {"op": "eq", "field": "region", "value": "north"})
            assert hits and all(h.metadata["region"] == "north" for h in hits)
            print("vector hits", [h.id for h in hits])

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "mongodb")

## 07_vespa

Source: `modules/16_landscape/examples/07_vespa.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("elasticsearch") is None or "vespa" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra landscape; start vespa and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Vespa tensor ranking and structured document retrieval.

    Run: uv run --extra landscape python modules/16_landscape/examples/07_vespa.py
    Concepts: Vespa tensor ranking and structured document retrieval.
    Expected: Vespa tensor score.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the vespa profile first.
    """

    # course-extra: landscape
    # course-profile: vespa
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Vespa tensor ranking and structured document retrieval.
        with session("vespa", 80) as (s, _, q):
            first = next(iter(s.catalog))
            result = s.db.get(s.path(first)).raise_for_status().json()
            print("native document", result["id"], "scope", result["fields"]["scope"])
            hits = s.search(q, 5, {"op": "eq", "field": "region", "value": "north"})
            assert hits and all(h.metadata["region"] == "north" for h in hits)
            print("vector hits", [h.id for h in hits])

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "vespa")

## 08_faiss_service

Source: `modules/16_landscape/examples/08_faiss_service.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("fastapi") is None or importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra apps --extra libs")
else:
    """Measure native FAISS retrieval through a real loopback HTTP service.

    Run: uv run --extra apps --extra libs python modules/16_landscape/examples/08_faiss_service.py
    Concepts: Measure native FAISS retrieval through a real loopback HTTP service.
    Expected: loopback HTTP timings.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: apps,libs
    import socket
    import tempfile
    import threading
    import time
    from pathlib import Path

    import httpx
    import uvicorn

    from common.apps.faiss_service import create_app
    from common.teaching import catalog, evidence, timed

    def main() -> None:
        # KEY IDEA: Measure native FAISS retrieval through a real loopback HTTP service.
        rows, matrix, q, meta = catalog(160)
        with tempfile.TemporaryDirectory() as directory, socket.socket() as listener:
            listener.bind(("127.0.0.1", 0))
            listener.listen()
            server = uvicorn.Server(
                uvicorn.Config(create_app(Path(directory), 384, meta["space"]), log_level="error")
            )
            thread = threading.Thread(
                target=server.run, kwargs={"sockets": [listener]}, daemon=True
            )
            thread.start()
            try:
                deadline = time.monotonic() + 20
                while not server.started:
                    if not thread.is_alive() or time.monotonic() > deadline:
                        raise RuntimeError("Local FAISS service failed to start")
                    time.sleep(0.05)
                with httpx.Client(base_url=f"http://127.0.0.1:{listener.getsockname()[1]}") as http:
                    http.put(
                        "/items",
                        json=[
                            {"id": p.id, "vector": v.tolist(), "text": p.title}
                            for p, v in zip(rows.itertuples(), matrix, strict=True)
                        ],
                    ).raise_for_status()

                    def query():
                        return (
                            http.post(
                                "/search",
                                json={"vector": q.tolist(), "space": meta["space"], "k": 5},
                            )
                            .raise_for_status()
                            .json()
                        )

                    hits = query()
                    assert len(hits) == 5
                    observation = timed(query)
                    http.post("/snapshot").raise_for_status()
                    print(hits, observation)
                    evidence(
                        "faiss-http",
                        {"loopback_http": observation, "ids": [h["id"] for h in hits]},
                        tier=3,
                    )
            finally:
                server.should_exit = True
                thread.join(10)

    if __name__ == "__main__":
        main()

## 09_managed_cost

Source: `modules/16_landscape/examples/09_managed_cost.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compare hypothetical managed price units without opening an account.

Run: uv run python modules/16_landscape/examples/09_managed_cost.py
Concepts: Compare hypothetical managed price units without opening an account.
Expected: hypothetical cost floor.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""


def estimate(storage_gib, months, reads, writes, price):
    metered = (
        storage_gib * months * price["storage"] + reads * price["read"] + writes * price["write"]
    )
    return max(months * price["monthly_minimum"], metered)


def main() -> None:
    # KEY IDEA: Compare hypothetical managed price units without opening an account.
    fictional = {"storage": 0.2, "read": 0.00001, "write": 0.00003, "monthly_minimum": 5}
    small = estimate(1, 1, 100, 100, fictional)
    large = estimate(10, 1, 1_000_000, 100_000, fictional)
    print("Illustrative dollars, not vendor quotes:", small, large)
    assert small == 5 and large > small


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

rows, _, _, _ = catalog(100)
print(rows.groupby("region").price.median())
# Change the group or price threshold, then repeat the native comparison for 16_landscape.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.